# AgentComet — Complete Feature Guide

This notebook demonstrates **all** features of the AgentComet SDK:

1. **Tools** — `@tool` decorator + builtin tools
2. **Agent Class** — Custom agents with `setup()`
3. **Memory & Conversations** — Auto-saved chat history + key-value store
4. **State Persistence** — Named checkpoints with rollback
5. **UAF Export & Load** — Portable agents that remember everything
6. **Local Server** — Seamlessly push/pull agents \n
7. **LLM Providers** — Ollama, OpenAI, Gemini, Anthropic, etc.

**Prerequisites:**
```bash
pip install uaf pyyaml requests
pip install -e .  # Install agentcomet
```

---

## Setup

In [1]:
import os
import shutil
from agentcomet import Agent, create_agent, load_agent
from agentcomet.models import Ollama
from agentcomet.tools import tool

llm = Ollama(model="gemma3:4b")
print("LLM ready:", llm.model)

LLM ready: gemma3:4b


---

## 1. Tools

The `@tool` decorator converts any function into a `ToolSpec` with auto-extracted name, description, and JSON schema.

In [2]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    return str(eval(expression))

print("Name:", multiply.name)
print("Description:", multiply.description)
print("Schema:", multiply.schema)

Name: multiply
Description: Multiplies two numbers together.
Schema: {'type': 'object', 'properties': {'a': {'type': 'integer', 'description': 'Parameter a'}, 'b': {'type': 'integer', 'description': 'Parameter b'}}, 'required': ['a', 'b']}


In [3]:
# Builtin tools
from agentcomet.tools import read, write

print("Builtin:", read.name, "-", read.description)
print("Builtin:", write.name, "-", write.description)

Builtin: read - Read contents of a file.
Builtin: write - Write content to a file.


---

## 2. Agent Class

Subclass `Agent`, configure in `setup()`, pass LLM at instantiation.

In [4]:
class MyAssistant(Agent):
    def setup(self):
        self.name = "assistant"
        self.description = "Personal assistant that remembers you"
        self.author = "Vaibhav"
        self.add_tools(multiply, calculator)

agent = MyAssistant(llm=llm)
print(agent.run("Hello! What can you do?"))

I can perform mathematical calculations using the `calculator` tool. I can also read and write files using the `read` and `write` tools, and multiply numbers using the `multiply` tool. Just let me know what you'd like me to do!



---

## 3. Memory & Conversations

Every agent has `self.memory`. When you call `agent.run()`, the conversation is **automatically saved** to `memory["messages"]`.

### 3A. Conversational memory — share info, agent remembers

In [5]:
# Share personal info with the agent
print(agent.run("Hi, my name is Vaibhav and my phone number is 9876543210"))

Hello Vaibhav! I have recorded your name and phone number.


In [6]:
print(agent.run("I work at AgentComet as a developer"))

Hello Vaibhav! I have recorded your name and phone number. I also now know that you work at AgentComet as a developer.


In [7]:
print(agent.run("Remember: project deadline is March 15th"))

Hello. I have recorded that the project deadline is March 15th.


In [8]:
# Now ask — the agent recalls from conversation history
print(agent.run("What is my name?"))

Vaibhav.


In [9]:
print(agent.run("What is my phone number?"))

Okay, Vaibhav. Your phone number is 9876543210.


In [9]:
# Check stored messages
msgs = agent.memory.get("messages", [])
print(f"\nConversation has {len(msgs)} messages:")
for msg in msgs:
    print(f"  [{msg['role']}] {msg['text'][:80]}")


Conversation has 10 messages:
  [user] Hello! What can you do?
  [agent] I can perform mathematical calculations using the `calculator` tool. I can also 
  [user] Hi, my name is Vaibhav and my phone number is 9876543210
  [agent] Hello Vaibhav! I have recorded your name and phone number.
  [user] I work at AgentComet as a developer
  [agent] Hello Vaibhav! I have recorded your name and phone number. I also now know that 
  [user] Remember: project deadline is March 15th
  [agent] Hello. I have recorded that the project deadline is March 15th.
  [user] What is my name?
  [agent] Vaibhav.


### 3B. Key-value memory — store structured data

In [10]:
# You can also store arbitrary key-value data
agent.memory.save("system_prompt", "You are a helpful personal assistant.")
agent.memory.save("preferences", {"theme": "dark", "lang": "en"})
agent.memory.save("notes", [
    "User prefers concise answers",
    "Deadline is March 15th"
])

print("All keys:", agent.memory.keys())
print("System prompt:", agent.memory.get("system_prompt"))
print("Notes:", agent.memory.get("notes"))
print()
print(agent.memory)  # Memory(N keys: [...])

All keys: ['messages', 'system_prompt', 'preferences', 'notes']
System prompt: You are a helpful personal assistant.
Notes: ['User prefers concise answers', 'Deadline is March 15th']

Memory(4 keys: ['messages', 'system_prompt', 'preferences', 'notes'])


---

## 4. State Persistence

Save memory snapshots — including full conversation history. Rollback anytime.

In [11]:
# Save current state (with all messages + data)
hash1 = agent.save_state()  # auto-hash
agent.save_state("after-intro")  # friendly name

[36e45fd5] State saved (4 keys)
[after-intro] State saved (4 keys)


'after-intro'

In [12]:
# Continue chatting — update info
print(agent.run("Actually, the deadline moved to March 20th"))
agent.memory.save("notes", ["Deadline updated to March 20th"])

agent.save_state("updated-deadline")

Okay, I have recorded that the project deadline is now March 20th.
[updated-deadline] State saved (4 keys)


'updated-deadline'

In [13]:
# View all checkpoints
agent.show_states()


States for 'assistant':
  [latest] 16ff9b13 (updated-deadline)  2026-03-29 13:28:38  (4 keys)
           a6a3e35c (after-intro)  2026-03-29 13:28:32  (4 keys)
           36e45fd5  2026-03-29 13:28:32  (4 keys)
           61431e10  2026-03-22 19:31:57  (4 keys)
           71882ee8  2026-03-22 19:31:52  (4 keys)
           e41d2df6  2026-03-22 19:31:52  (4 keys)



[{'hash': '16ff9b13', 'created_at': '2026-03-29 13:28:38', 'key_count': 4},
 {'hash': 'a6a3e35c', 'created_at': '2026-03-29 13:28:32', 'key_count': 4},
 {'hash': '36e45fd5', 'created_at': '2026-03-29 13:28:32', 'key_count': 4},
 {'hash': '61431e10', 'created_at': '2026-03-22 19:31:57', 'key_count': 4},
 {'hash': '71882ee8', 'created_at': '2026-03-22 19:31:52', 'key_count': 4},
 {'hash': 'e41d2df6', 'created_at': '2026-03-22 19:31:52', 'key_count': 4}]

In [14]:
# Rollback to before deadline change
agent.load_state("after-intro")
print("After rollback:")
print("  Notes:", agent.memory.get("notes"))
print("  Messages:", len(agent.memory.get("messages", [])), "messages")

# Ask about deadline — should reflect the ORIGINAL date
print(agent.run("When is the project deadline?"))

Loaded state 'after-intro' (4 keys)
After rollback:
  Notes: ['User prefers concise answers', 'Deadline is March 15th']
  Messages: 10 messages
I apologize, but I encountered an error while trying to retrieve the project deadline from the conversation history. The file 'conversation_history.txt' does not exist. I have recorded that the project deadline is March 15th.


---

## 5. UAF Export & Load — Agent Remembers After Reload

Export to `.uaf` — conversation + memory auto-packed.  
Load later — agent picks up right where you left off.

In [15]:
# Reset to latest state (has all the info)
agent.load_state("updated-deadline")

# Export — everything auto-packed
agent.export("my_assistant.uaf")
print("Exported!")

Loaded state 'updated-deadline' (4 keys)
Exported AgentComet agent 'assistant' to my_assistant.uaf
Exported!


In [16]:
# Simulate a fresh session — load from file
loaded = load_agent("my_assistant.uaf")

print("Loaded agent type:", type(loaded))
print(f"Restored {len(loaded.memory.get('messages', []))} messages")
print("Restored notes:", loaded.memory.get("notes"))
print()

# Ask the loaded agent about info from BEFORE the export
print("--- Asking loaded agent about stored info ---")
print(loaded.run("What is my name?"))
print(loaded.run("What is my phone number?"))
print(loaded.run("When is the project deadline?"))

Loaded agent type: <class 'agentcomet.agents.factory.create_agent.<locals>.DynamicAgent'>
Restored 12 messages
Restored notes: ['Deadline updated to March 20th']

--- Asking loaded agent about stored info ---
Vaibhav.
I don’t have your phone number recorded. I only have your name, Vaibhav, and that you work at AgentComet as a developer.
The project deadline is March 20th.


In [17]:
# Cleanup
os.remove("my_assistant.uaf")

---

## 6. Local Server Interaction

Sync your agents with a locally hosted AgentComet server effortlessly. The SDK automates generating the portable UAF and determining the next semantic version.


In [19]:
from agentcomet import Settings, Agent

# Configure connection programmatically (or use environment variables)
Settings.init(
    AGENTCOMET_LOCAL_URL="http://localhost:3451",
    AGENTCOMET_LOCAL_KEY="your-local-key"
)

try:
    # Push the agent. UAF is created dynamically, and version is auto-incremented!
    push_res = agent.push_local(version="auto")
    print("Push response:", push_res)
    
    # Pull the agent from the server. It downloads the UAF and loads it instantly!
    downloaded_agent = Agent.pull_local("assistant", version="latest")
    print("Downloaded agent:", downloaded_agent.name)
except Exception as e:
    print("Local Server Interaction Error:", e)


Exported AgentComet agent 'assistant' to C:\Users\Vaibh\AppData\Local\Temp\tmpo2hifpuj.uaf
Successfully pushed 'assistant' v0.1.0 to http://localhost:3451
Push response: {'agent': {'id': 'agt_804b72b304f546229f6e3f3022a3d745', 'name': 'assistant', 'slug': 'assistant', 'description': 'Personal assistant that remembers you', 'visibility': 'private', 'readme': 'Personal assistant that remembers you', 'created_at': '2026-03-29T08:02:23.752Z', 'updated_at': '2026-03-29T08:02:23.759Z', 'last_published_at': None, 'latest_version': '0.1.0', 'latest_version_created_at': '2026-03-29T08:02:23.759Z', 'version_count': 1}, 'version': {'id': 'ver_7a2e5ffafba1470484780e302acc041b', 'agent_id': 'agt_804b72b304f546229f6e3f3022a3d745', 'version': '0.1.0', 'notes': None, 'metadata_json': None, 'artifact_path': 'D:\\Projects\\Personal\\DefaultLoop\\AgentComet-UI\\data\\agents\\agt_804b72b304f546229f6e3f3022a3d745\\versions\\version_1774771343756\\assistant.uaf', 'artifact_file_name': 'assistant.uaf', 'arti

---

## 7. LLM Providers

Pass LLM instances directly to agents.

In [ ]:
from agentcomet.models import Ollama, OpenAIChat, Gemini, Anthropic, OpenRouter, Perplexity

# Ollama (local)
ollama = Ollama(model="gemma3:4b")
result = ollama.generate("What is the capital of France?")
print("Direct call:", result[:100])

# Other providers (require API keys)
# openai = OpenAIChat(model="gpt-4o")
# gemini = Gemini(model="gemini-1.5-flash")
# claude = Anthropic(model="claude-3-5-sonnet")

---

## Cleanup

In [ ]:
if os.path.exists(".agentcomet"):
    shutil.rmtree(".agentcomet")
    print("Cleaned up .agentcomet/")

---

## Summary

| Feature | API | Description |
|---------|-----|-------------|
| **Tools** | `@tool` | Auto ToolSpec from functions |
| **Agent** | `class MyAgent(Agent)` | Full control via `setup()` |
| **Declarative** | `create_agent(...)` | One-liner agents |
| **Auto Memory** | `agent.run("...")` | Chat auto-saved to messages |
| **Key-Value** | `memory.save(k, v)` | Store anything |
| **Save State** | `save_state("name")` | Named or hashed checkpoint |
| **Load State** | `load_state("name")` | Rollback to any checkpoint |
| **Export** | `agent.export("file.uaf")` | Memory auto-packed |
| **Load** | `load_agent("file.uaf")` | Agent remembers everything |
| **Local Sync** | `push_local()`, `pull_local()` | Sync with local server |
| **LLM** | `Ollama(model=...)` | 6+ providers, pass directly |